In [1]:
%load_ext autoreload
%autoreload 2

import sys, os, json
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches

In [2]:
sys.path.append(os.path.abspath('../src'))
from config.vessel_specs import POWER_CONFIG, DELAY_PARAMS
from orchestrator import KiaraOrchestrator

In [3]:
with open('../src/config/schedule.json', 'r') as f:
    cyclades_schedule = json.load(f)
with open('../src/config/weather_params.json', 'r') as f:
    climate_db = json.load(f)

In [ ]:
# --- 1. UI WIDGETS (UPDATED LAYOUT) ---
style = {'description_width': '220px'}
# Force the widgets to be wide enough to show both the long label and the slider bar
wide_layout = widgets.Layout(width='450px') 

# A. Environment Controls
month_dd = widgets.Dropdown(options=[('January (Winter)', '1'), ('May (Spring)', '5'), ('August (Summer Meltemi)', '8'), ('November (Autumn)', '11')], value='8', description='Season:', style=style, layout=wide_layout)
w0_slider = widgets.FloatSlider(value=0.2, min=0.0, max=1.0, step=0.05, description='Initial Weather Hazard (W0):', style=style, layout=wide_layout)
rng_seed = widgets.IntSlider(value=42, min=1, max=1000, description='RNG Seed (Lock Weather):', style=style, layout=wide_layout)

# B. Physics & Noise
wave_res = widgets.FloatSlider(value=0.15, min=0.0, max=0.40, step=0.05, description='Wave Resistance Penalty (%):', style=style, layout=wide_layout)
main_noise = widgets.IntSlider(value=500, min=100, max=1500, step=50, description='Main Engine Noise (kW):', style=style, layout=wide_layout)
aux_noise = widgets.IntSlider(value=15, min=0, max=50, step=1, description='Hotel Load Noise (kW):', style=style, layout=wide_layout)

# C. Operations & Human Element
maneuver_time = widgets.FloatSlider(value=6.0, min=2.0, max=15.0, step=0.5, description='Avg Maneuver Time (mins):', style=style, layout=wide_layout)
thruster_start = widgets.FloatSlider(value=0.15, min=0.05, max=0.40, step=0.01, description='Thruster ON Threshold:', style=style, layout=wide_layout)
thruster_max = widgets.FloatSlider(value=0.30, min=0.10, max=0.80, step=0.01, description='Thruster MAX Threshold:', style=style, layout=wide_layout)

# D. Congestion & Delays
port_mu = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, description='Port Delay Avg (μ):', style=style, layout=wide_layout)
transit_mu = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, description='Transit Delay Avg (μ):', style=style, layout=wide_layout)
weather_delay = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, description='Storm Delay Penalty (+μ):', style=style, layout=wide_layout)

run_button = widgets.Button(description='▶ Run Digital Twin', button_style='success', icon='play', layout=widgets.Layout(width='300px', height='40px'))
output = widgets.Output()

In [5]:
# --- ADVANCED PLOTTING FUNCTION ---
def plot_dashboard_trace(df_micro, df_timeline, window_minutes=240):
    df_slice = df_micro.head(window_minutes * 60)
    max_time = df_slice['timestamp'].max()
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12), sharex=True, gridspec_kw={'height_ratios': [1.5, 3, 2]})
    
    # --- Meso Weather ---
    ax1.plot(df_slice['timestamp'], df_slice['W_global'], color='purple', linewidth=2)
    ax1.axhline(thruster_start.value, color='red', linestyle='--', alpha=0.5, label="Thruster Threshold")
    ax1.set_ylabel('Weather Hazard [0-1]', fontweight='bold')
    ax1.legend(loc="upper right")
    ax1.grid(True, alpha=0.3)
    ax1.set_title("MARINER Digital Twin: Expert Tuning Dashboard", fontsize=16, fontweight='bold')

    # --- Power Graphs & Dynamic Baselines ---
    state_to_main = {'transit': POWER_CONFIG['P_main_sea'], 'maneuvering': POWER_CONFIG['P_main_maneuver'], 'port_dwell': POWER_CONFIG['P_main_port']}
    state_to_aux = {'transit': POWER_CONFIG['P_aux_sea'], 'maneuvering': POWER_CONFIG['P_aux_maneuver'], 'port_dwell': POWER_CONFIG['P_aux_port']}
    
    # Calculate Nominal Baselines as step functions
    nominal_main = df_slice['state'].map(state_to_main).fillna(0)
    nominal_aux = df_slice['state'].map(state_to_aux).fillna(0)

    # Main Engine
    ax2.plot(df_slice['timestamp'], df_slice['P_main_kW'], color='#d62728', alpha=0.3, label='Raw Power')
    ax2.plot(df_slice['timestamp'], df_slice['P_main_kW'].rolling(60, center=True).mean(), color='darkred', linewidth=2, label='1-Min Avg')
    ax2.plot(df_slice['timestamp'], nominal_main, color='black', linestyle='--', linewidth=1.5, label="Nominal Baseline", drawstyle='steps-post')
    ax2.set_ylabel('Main Power [kW]', fontweight='bold')
    
    # Aux Engine
    ax3.plot(df_slice['timestamp'], df_slice['P_aux_kW'], color='#2ca02c', label="Electric Load (Hotel + Thrusters)")
    ax3.plot(df_slice['timestamp'], nominal_aux, color='black', linestyle='--', linewidth=1.5, label="Nominal Hotel Baseline", drawstyle='steps-post')
    ax3.set_ylabel('Aux Power [kW]', fontweight='bold')
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    
    # --- Background Colors & Delay Hatching ---
    color_map = {'port_dwell': '#ff7f0e', 'maneuvering': '#9467bd', 'transit': '#1f77b4'}
    
    for _, row in df_timeline.iterrows():
        if row['start_time'] > max_time: continue
        state_color = color_map.get(row['state'], 'white')
        end_t = min(row['end_time'], max_time)
        
        # Apply background to BOTH graphs
        ax2.axvspan(row['start_time'], end_t, color=state_color, alpha=0.1)
        ax3.axvspan(row['start_time'], end_t, color=state_color, alpha=0.1)
        
        # Visualize Delays (Red Hatching at the end of the state)
        if row.get('delay_mins', 0) > 0:
            nominal_end = row['end_time'] - timedelta(minutes=row['delay_mins'])
            if nominal_end < max_time:
                ax2.axvspan(nominal_end, end_t, hatch='///', edgecolor='red', facecolor='none', alpha=0.3)
                ax3.axvspan(nominal_end, end_t, hatch='///', edgecolor='red', facecolor='none', alpha=0.3)

    # Formatting & Legends
    for ax in [ax2, ax3]:
        ax.grid(True, alpha=0.3)
        handles, labels = ax.get_legend_handles_labels()
        # Add background legend patches
        handles.extend([
            mpatches.Patch(color='#1f77b4', alpha=0.2, label='Transit Phase'),
            mpatches.Patch(color='#9467bd', alpha=0.2, label='Maneuvering Phase'),
            mpatches.Patch(color='#ff7f0e', alpha=0.2, label='Port Dwell'),
            mpatches.Patch(hatch='///', edgecolor='red', facecolor='none', label='Stochastic Delay Overrun')
        ])
        ax.legend(handles=handles, loc="upper right")
    
    plt.tight_layout()
    plt.show()

# --- EXECUTION LOGIC ---
def execute(b):
    with output:
        clear_output(wait=True)
        print("Initializing digital twin... Applying physics and generating climate...")
        
        # 1. Lock RNG for reproducibility
        np.random.seed(rng_seed.value)
        
        # 2. Inject UI Parameters
        cfg = POWER_CONFIG.copy()
        cfg['sigma_main_base'] = main_noise.value
        cfg['sigma_aux_base'] = aux_noise.value
        cfg['wave_resistance_factor'] = wave_res.value
        cfg['thruster_weather_threshold'] = thruster_start.value
        cfg['thruster_max_weather'] = thruster_max.value
        
        dly = DELAY_PARAMS.copy()
        dly['maneuvering']['avg_mins'] = maneuver_time.value
        dly['port']['mu'] = port_mu.value
        dly['transit']['mu'] = transit_mu.value
        dly['weather_penalty'] = weather_delay.value
        
        orch = KiaraOrchestrator(
            schedule=cyclades_schedule,
            start_dt=datetime(2026, int(month_dd.value), 3, 7, 0),
            delay_params=dly, 
            power_config=cfg,
            weather_params=climate_db[month_dd.value]["jacobi"],
            k_factors=climate_db[month_dd.value]["k_factors"],
            initial_weather=w0_slider.value
        )
        
        df_macro, df_micro = orch.run_simulation(days=1)
        plot_dashboard_trace(df_micro, df_macro, window_minutes=600) # Show 10 hours

run_button.on_click(execute)

In [6]:
col_layout = widgets.Layout(width='50%', padding='0px 20px 0px 0px')

ui = widgets.VBox([
    widgets.HTML("<h2>⚙️ MARINER Configuration Panel</h2>"),
    widgets.HBox([
        widgets.VBox([widgets.HTML("<b> 1. Environment & Climate</b>"), month_dd, w0_slider, rng_seed], layout=col_layout),
        widgets.VBox([widgets.HTML("<b> 2. Vessel Physics</b>"), wave_res, main_noise, aux_noise], layout=col_layout)
    ]),
    widgets.HTML("<hr>"),
    widgets.HBox([
        widgets.VBox([widgets.HTML("<b> 3. Operations & Human Element</b>"), maneuver_time, thruster_start, thruster_max], layout=col_layout),
        widgets.VBox([widgets.HTML("<b> 4. Congestion & Delays</b>"), port_mu, transit_mu, weather_delay], layout=col_layout)
    ]),
    widgets.HTML("<br>"),
    run_button,
    output
])
display(ui)